# Q21: качество данных, физическая интерпретация и контролируемый эксперимент
Описание справочника: «расход газа поддува на входе К-201». Наблюдаемые значения и связь с ЛИМС вызывают сомнения в этом описании. Физический смысл не переопределяется автоматически.

Эксперимент: пять вариантов, два периода, три seeds. Гиперпараметры фиксированы по ранее выбранной модели риска h0. Счётчики zero/negative/unchanged удалены из ВСЕХ вариантов, чтобы Q21 не оставался в них косвенно. Данные оценки ранее изучались; это диагностический эксперимент.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display
HERE=Path.cwd().resolve()
EDA=HERE if HERE.name=='eda' else HERE/'eda'
A=EDA/'artifacts/q21'
R=EDA/'experiments/q21_ablation_20260915'
def show(fig,name):
    fig.write_html(A/f'{name}.html',include_plotlyjs=True)
    fig.show()
display(pd.read_csv(A/'profile.csv'))
display(pd.read_csv(A/'constant_episodes.csv').head(15))

,Unnamed: 0,value
0,count,189217.000000
1,mean,17.633817
2,std,50.721212
3,min,-0.099167
4,1%,3.047407
5,5%,5.368042
6,25%,7.548551
7,50%,8.514394
8,75%,9.454333
9,95%,24.827576


,start,end,value,n,hours
0,2024-03-18 14:20:00,2024-04-18 10:20:00,307.0,4441,740.166667
1,2026-04-22 14:30:00,2026-04-27 21:30:00,307.0,763,127.166667
2,2026-03-15 07:30:00,2026-03-16 09:20:00,307.0,156,26.000000
3,2026-01-18 09:10:00,2026-01-19 10:30:00,307.0,153,25.500000
4,2026-06-23 09:10:00,2026-06-23 13:50:00,307.0,29,4.833333
5,2026-06-18 10:40:00,2026-06-18 14:20:00,307.0,23,3.833333
6,2025-12-09 19:40:00,2025-12-09 21:50:00,307.0,14,2.333333
7,2025-10-08 13:20:00,2025-10-08 14:50:00,307.0,10,1.666667
8,2025-12-10 09:40:00,2025-12-10 10:50:00,307.0,8,1.333333


In [2]:
monthly=pd.read_csv(A/'monthly.csv',parse_dates=['date'])
show(px.line(monthly,x='date',y='median',markers=True,title='Месячная медиана Q21'),'q21_monthly_median')
show(px.bar(monthly,x='date',y='fraction_above50',title='Доля значений Q21=307 по месяцам'),'q21_invalid_monthly')

In [3]:
lab=pd.read_csv(A/'lims_matches.csv',parse_dates=['timestamp'])
valid=lab.Q21.between(0,50)
show(px.scatter(lab[valid],x='Q21',y='value',facet_col='year',hover_data=['timestamp'],title='Q21 и лабораторная сера: диагностическая маска Q21 от 0 до 50'),'q21_lims_scatter')
display(pd.read_csv(A/'yearly_lims_metrics.csv'))
daily=pd.read_csv(A/'daily_pak_q21.csv',parse_dates=['timestamp'])
show(px.line(daily,x='timestamp',y=['value','Q21'],title='Дневные медианы ПАК серы и Q21; value — ПАК'),'q21_pak_timeline')
display(pd.read_csv(A/'pak_comparison.csv'))

,year,mask,n,spearman,mae,median_ae,bias_q21_minus_lab
0,2023,all,406,0.710578,1.025125,0.715387,-0.510361
1,2023,diagnostic_Q21_0_50,406,0.710578,1.025125,0.715387,-0.510361
2,2024,all,386,0.597763,7.911910,0.898750,-4.357466
3,2024,diagnostic_Q21_0_50,384,0.613713,7.140407,0.894540,-5.149221
4,2025,all,417,0.568191,1.556200,0.657715,-0.139272
5,2025,diagnostic_Q21_0_50,416,0.578823,1.531553,0.655066,-0.111220
6,2026,all,249,0.601248,12.080502,0.863720,10.869756
7,2026,diagnostic_Q21_0_50,240,0.639981,1.354771,0.791663,0.098622


,mask,n,spearman,mae,median_ae,near_equal_fraction
0,all,189218,0.424845,10.301216,0.88633,0.0
1,diagnostic_Q21_0_50,183587,0.468218,1.499440,0.84733,0.0


In [4]:
m=pd.read_csv(R/'metrics.csv')
display(m.groupby(['variant','year']).agg(n_seeds=('seed','size'),ap_mean=('ap','mean'),ap_min=('ap','min'),ap_max=('ap','max'),recall_mean=('recall','mean'),fpr_mean=('fpr','mean')))
show(px.box(m,x='variant',y='ap',facet_col='year',points='all',title='Контролируемое исключение Q21: AP по трём seeds'),'q21_ablation_ap')
show(px.scatter(m,x='fpr',y='recall',color='variant',facet_col='year',hover_data=['seed','threshold'],title='Пропуски и ложные тревоги: порог выбран на calibration'),'q21_ablation_tradeoff')

n_seeds   ap_mean    ap_min    ap_max  \
variant                 year                                          
full                    2024        3  0.212904  0.197314  0.230160   
                        2026        3  0.378212  0.369285  0.391916   
q21_delayed6h           2024        3  0.116662  0.110281  0.120378   
                        2026        3  0.219792  0.205880  0.228280   
q21_only                2024        3  0.242133  0.236575  0.249331   
                        2026        3  0.491609  0.474104  0.505283   
without_q21             2024        3  0.121448  0.111377  0.126813   
                        2026        3  0.217845  0.197607  0.229114   
without_q21_without_lab 2024        3  0.114290  0.101539  0.122292   
                        2026        3  0.207500  0.203619  0.213337   

                              recall_mean  fpr_mean  
variant                 year                         
full                    2024     0.233333  0.116105  
                        2026     0.438095  0.098413  
q21_delayed6h           2024     0.000000  0.041199  
                        2026     0.257143  0.122222  
q21_only                2024     0.500000  0.179775  
                        2026     0.409524  0.071429  
without_q21             2024     0.000000  0.037453  
                        2026     0.285714  0.166667  
without_q21_without_lab 2024     0.000000  0.041199  
                        2026     0.285714  0.169841

In [5]:
base=m[m.variant=='full'][['year','seed','ap']].rename(columns={'ap':'ap_full'})
paired=m.merge(base,on=['year','seed'])
paired['delta_ap']=paired.ap-paired.ap_full
display(paired[['variant','year','seed','ap','delta_ap']])
p=pd.read_csv(R/'predictions.csv',parse_dates=['timestamp'])
assert not p.duplicated(['variant','year','seed','timestamp']).any()
assert m.groupby(['variant','year']).size().eq(3).all()
assert len(m)==30
print('30 моделей; сравнения на одинаковых датах и seeds.')

,variant,year,seed,ap,delta_ap
0,full,2024,17,0.197314,0.000000
1,full,2024,42,0.230160,0.000000
2,full,2024,2026,0.211237,0.000000
3,full,2026,17,0.369285,0.000000
4,full,2026,42,0.391916,0.000000
5,full,2026,2026,0.373434,0.000000
6,without_q21,2024,17,0.126153,-0.071161
7,without_q21,2024,42,0.126813,-0.103347
8,without_q21,2024,2026,0.111377,-0.099859
9,without_q21,2026,17,0.197607,-0.171678


30 моделей; сравнения на одинаковых датах и seeds.


## Ограничения интерпретации
307 — вероятный код недостоверности/предельное значение, но это требует подтверждения. Фильтр 0–50 используется лишь для диагностики; это не технологический предел. Сравнения на отфильтрованных наблюдениях нельзя напрямую сравнивать с метриками всех наблюдений.

Вариант q21_only использует Q21 и его прошлые агрегаты, без прошлых ЛИМС. В q21_delayed6h только признаки Q21 сдвинуты назад на 6 ч; остальные признаки текущие. Разница не равна физическому лагу процесса. Полный вывод: Q21_FINDINGS.md.